# YOLOv11 Small Object Detection — xView (Paper Replication)
### Yuan et al. (2026), ESWA 307 · 6-Fold Cross-Validation

---

| Fix | What changed from previous version |
|---|---|
| Model | Default `yolo11m.pt` (~20M params, paper: 25.37M) |
| CV | 6-fold cross-validation with mean ± std reporting |
| Iterations | Computed from dataset size, not assumed 100 epochs |
| Early stopping | Disabled in paper mode (patience=999) |
| Split | Source-image-level folds, no data leakage |
| Channels | Documented: RGB only (YOLO requires 3ch) |
| Data | Only train_images used (val_images has no annotations) |

### Paper results (Table 13 — xView, 6-fold CV, mean ± std)
| Metric | YOLOv11 |
|---|---|
| AP50 | 69.47 (±3.07) |
| AP50:95 | 29.58 (±1.40) |
| F1 | 73.97 (±1.81) |


---
## 1. Configuration

### Quick start
1. Set `DEBUG_MODE = True` → runs 1 epoch on fold 0 (~3 min)
2. Set `DEBUG_MODE = False` → runs full paper protocol
3. On Kaggle: attach `hassanmojab/xview-dataset` via sidebar


In [29]:
# ══════════════════════════════════════════════════════════════════
#  1.1  EDIT THESE VARIABLES
# ══════════════════════════════════════════════════════════════════

# ---- Mode ----
DEBUG_MODE = False          # True=1 epoch, False=full paper protocol

# ---- Environment ----
RUN_ENV = "kaggle"         # "kaggle" | "colab" | "local"
DATASET_HANDLE = "hassanmojab/xview-dataset"
LOCAL_DATASET_DIR = "./data/xview"
LOCAL_KAGGLE_JSON = "~/.kaggle/kaggle.json"
MOUNT_GDRIVE = True
GDRIVE_OUTPUT_DIR = "MyDrive/xview_outputs"

# ---- Cross-validation (Paper: 6-fold) ----
NUM_FOLDS    = 6           # Paper Section 4.1
CURRENT_FOLD = 0           # Which fold to train (0-5), or "all"
                           # On Kaggle, run one fold per session (time limit)

# ---- Model (Paper Table 14: YOLOv11 = 25.37M params) ----
YOLO_VARIANT = "yolo11m.pt"  # Paper-scale. Use yolo11n.pt for DEBUG only.

# ---- Paper hyperparameters (Table 5) ----
TILE_SIZE      = 512
TRAIN_RATIO    = 0.70      # used WITHIN each fold: not needed with CV
BATCH_SIZE     = 8
LEARNING_RATE  = 1.01e-4
MOMENTUM       = 0.89
WEIGHT_DECAY   = 5e-4
IMG_SIZE       = 512
FLIP_PROB      = 0.5
CONF_THRESHOLD = 0.25
SEED           = 42

# ---- Small-object definition (Paper Section 3.1) ----
MAX_OBJ_PX = 1000
MIN_OBJ_PX = 10

# ---- Behaviour ----
FORCE_REDOWNLOAD = False
RESUME_TRAINING  = True
SKIP_TRAINING    = False

# ══════════════════════════════════════════════════════════════════
#  Compute paper-matched training budget
# ══════════════════════════════════════════════════════════════════
# Paper Table 5: YOLOv11 = 94K iterations
# Paper Table 4: 7545 training images per fold
# iterations_per_epoch = ceil(n_train / batch_size)
# epochs = total_iterations / iterations_per_epoch
# We compute this AFTER tiling; for now set a placeholder
PAPER_TOTAL_ITERS = 94_000
EPOCHS = 100  # will be recalculated after seeing actual dataset size

# ══════════════════════════════════════════════════════════════════
#  DEBUG overrides
# ══════════════════════════════════════════════════════════════════
if DEBUG_MODE:
    EPOCHS       = 1
    BATCH_SIZE   = 4
    YOLO_VARIANT = "yolo11n.pt"  # fast for testing
    if isinstance(CURRENT_FOLD, str):
        CURRENT_FOLD = 0
    print("=" * 60)
    print("  DEBUG MODE: 1 epoch, yolo11n, fold 0 only")
    print("=" * 60)
else:
    print("=" * 60)
    print(f"  PAPER MODE: {YOLO_VARIANT}, {NUM_FOLDS}-fold CV")
    print(f"  Fold(s): {CURRENT_FOLD}")
    print("=" * 60)

print(f"  RUN_ENV  : {RUN_ENV}")
print(f"  Model    : {YOLO_VARIANT}")


  PAPER MODE: yolo11m.pt, 6-fold CV
  Fold(s): 0
  RUN_ENV  : kaggle
  Model    : yolo11m.pt


---
## 2. Environment Setup


In [30]:
# ══════════════════════════════════════════════════════════════════
#  2.1  Validate + install + import + GPU + dirs
# ══════════════════════════════════════════════════════════════════
import os, sys, platform, subprocess, shutil, zipfile, time
from pathlib import Path

print("[2.1] Setting up environment ...")
assert RUN_ENV in ("kaggle","colab","local"), f"Bad RUN_ENV: {RUN_ENV}"

# Install
def _pip(p, n=None):
    try: __import__(p)
    except ImportError:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",n or p],
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for p,n in [("numpy",None),("pandas",None),("matplotlib",None),("seaborn",None),
            ("cv2","opencv-python-headless"),("tqdm",None),("sklearn","scikit-learn"),
            ("yaml","pyyaml"),("torch",None),("torchvision",None),
            ("ultralytics",None),("kaggle",None)]:
    _pip(p,n)

import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns, cv2, json, random, warnings, yaml
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
from datetime import datetime
import torch, torchvision
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize":(12,6),"figure.dpi":100,"font.size":11})
sns.set_style("whitegrid")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
from ultralytics import YOLO

# GPU
HW = {"device":"cpu","name":"CPU","count":0,"mem_gb":0}
if torch.cuda.is_available():
    HW.update(device="cuda", count=torch.cuda.device_count(),
              name=torch.cuda.get_device_name(0),
              mem_gb=round(torch.cuda.get_device_properties(0).total_memory/1e9,1))
elif hasattr(torch.backends,"mps") and torch.backends.mps.is_available():
    HW.update(device="mps", name="Apple MPS", count=1)
print(f"  GPU: {HW['name']}  ({HW['mem_gb']} GB)")
if HW["device"]=="cpu":
    print("  WARNING: No GPU — training will be extremely slow")

# Dirs
def _build_paths():
    if RUN_ENV=="kaggle": base,data=Path("/kaggle/working"),Path("/kaggle/working/xview_data")
    elif RUN_ENV=="colab": base,data=Path("/content"),Path("/content/xview_data")
    else: base,data=Path("."),Path(LOCAL_DATASET_DIR).expanduser().resolve()
    out = base/"outputs"
    if RUN_ENV=="colab" and MOUNT_GDRIVE:
        try:
            from google.colab import drive; drive.mount("/content/drive",force_remount=False)
            out = Path("/content/drive")/GDRIVE_OUTPUT_DIR
        except: pass
    P = {"DATA":data,"OUTPUT":out,"CKPT":out/"checkpoints","PLOTS":out/"plots",
         "METRICS":out/"metrics","TILES":out/"tiles","YOLO_DS":out/"yolo_dataset"}
    for k,d in P.items():
        if k!="DATA": d.mkdir(parents=True,exist_ok=True)
    return P

P = _build_paths()
print(f"  Output: {P['OUTPUT']}")
print("[2.1] Done.")


[2.1] Setting up environment ...
  GPU: Tesla T4  (15.6 GB)
  Output: /kaggle/working/outputs
[2.1] Done.


---
## 3. Dataset Loading

### Data split rationale
The Kaggle xView dataset contains:
- `train_images/` + GeoJSON annotations → **labelled, we use this**
- `val_images/` → **NO public annotations** → cannot use for training/evaluation

We use **only the labelled training data**, tile it, then create our own
train/val splits via 6-fold CV at the source-image level (no data leakage).

### Channel handling
The raw xView imagery is 8-band WorldView-3. The Kaggle repackaging stores
standard TIF files that OpenCV reads as 3-channel (RGB). This is the same
input format YOLO expects.  The paper's methods also operate on standard
image channels — 8-band processing would require custom architectures.


In [31]:
# ══════════════════════════════════════════════════════════════════
#  3.1  Dataset discovery
# ══════════════════════════════════════════════════════════════════
print("[3.1] Finding dataset ...")

from pathlib import Path
import subprocess
import zipfile

def safe_list_dirs(root: Path):
    dirs = []
    try:
        for p in root.rglob("*"):
            if p.is_dir():
                dirs.append(p)
    except Exception:
        pass
    return dirs

def find_xview(search_root):
    root = Path(search_root)

    empty = {
        "valid": False,
        "geojson": None,
        "images_dir": None,
        "labels_dir": None,
    }

    if not root.exists():
        return empty

    # include root + all nested directories
    all_dirs = [root] + safe_list_dirs(root)

    geo = None
    img = None
    lbl = None

    # ---------- Find geojson ----------
    geo_patterns = [
        "*train*.geojson",
        "*xview*.geojson",
        "*xView*.geojson",
        "*.geojson",
        "*.json",
    ]

    for pattern in geo_patterns:
        for p in root.rglob(pattern):
            if p.is_file():
                geo = p
                break
        if geo is not None:
            break

    # ---------- Find image directory ----------
    preferred_image_dirnames = {
        "train_images", "images", "train", "imgs", "image", "jpegimages"
    }

    # first: prefer common image folder names
    for d in all_dirs:
        try:
            if d.name.lower() in preferred_image_dirnames:
                has_img = any(d.glob("*.tif")) or any(d.glob("*.png")) or any(d.glob("*.jpg")) or any(d.glob("*.jpeg"))
                if has_img:
                    img = d
                    break
        except Exception:
            pass

    # second: any folder containing image files
    if img is None:
        for d in all_dirs:
            try:
                has_img = any(d.glob("*.tif")) or any(d.glob("*.png")) or any(d.glob("*.jpg")) or any(d.glob("*.jpeg"))
                if has_img:
                    img = d
                    break
            except Exception:
                pass

    # ---------- Find labels directory ----------
    preferred_label_dirnames = {
        "train_labels", "labels", "label", "annotations", "annots", "anno"
    }

    for d in all_dirs:
        try:
            if d.name.lower() in preferred_label_dirnames:
                has_lbl = (
                    any(d.glob("*.txt")) or
                    any(d.glob("*.geojson")) or
                    any(d.glob("*.json"))
                )
                if has_lbl:
                    lbl = d
                    break
        except Exception:
            pass

    # fallback: directory that contains txt/json label-like files
    if lbl is None:
        for d in all_dirs:
            try:
                has_lbl = (
                    any(d.glob("*.txt")) or
                    any(d.glob("*.geojson")) or
                    any(d.glob("*.json"))
                )
                if has_lbl:
                    lbl = d
                    break
            except Exception:
                pass

    # if geojson found but labels_dir not found, use geojson parent
    if geo is not None and lbl is None:
        lbl = geo.parent

    valid = (img is not None) and ((geo is not None) or (lbl is not None))

    return {
        "valid": valid,
        "geojson": geo,
        "images_dir": img,
        "labels_dir": lbl,
    }

def scan_kaggle_input():
    ki = Path("/kaggle/input")
    if not ki.exists():
        return None

    print(f"  Scanning {ki} ...")

    # print all top-level folders so you can verify what Kaggle mounted
    top_dirs = [p for p in ki.iterdir() if p.is_dir()]
    if not top_dirs:
        print("  No directories found inside /kaggle/input")
    else:
        print("  Top-level folders found:")
        for p in top_dirs:
            print(f"    - {p}")

    # 1) try each top-level mounted dataset folder
    for sub in sorted(top_dirs):
        print(f"    Checking {sub} ...")
        info = find_xview(sub)
        if info["valid"]:
            print("    MATCH FOUND")
            print(f"      Images : {info['images_dir']}")
            print(f"      Labels : {info['labels_dir']}")
            print(f"      GeoJSON: {info['geojson']}")
            return sub, info

    # 2) fallback: search the whole /kaggle/input recursively
    print("  No match at top level. Searching entire /kaggle/input recursively ...")
    info = find_xview(ki)
    if info["valid"]:
        print("    MATCH FOUND in recursive scan")
        print(f"      Images : {info['images_dir']}")
        print(f"      Labels : {info['labels_dir']}")
        print(f"      GeoJSON: {info['geojson']}")
        return ki, info

    return None


DATA_DIR = None
DS_INFO = None

if RUN_ENV == "kaggle":
    result = scan_kaggle_input()
    if result:
        DATA_DIR, DS_INFO = result
    else:
        print("  !!! Dataset not found in /kaggle/input/")
        print("  !!! Please inspect mounted folders manually with:")
        print("      !find /kaggle/input -maxdepth 4 -type d")
        raise FileNotFoundError("Could not locate the attached xView dataset inside /kaggle/input")
else:
    info = find_xview(P["DATA"])
    if info["valid"]:
        DATA_DIR, DS_INFO = P["DATA"], info
    else:
        print("  Downloading via Kaggle API ...")
        P["DATA"].mkdir(parents=True, exist_ok=True)

        subprocess.run(
            ["kaggle", "datasets", "download", "-d", DATASET_HANDLE, "-p", str(P["DATA"]), "--unzip"],
            check=True
        )

        for zp in P["DATA"].rglob("*.zip"):
            try:
                with zipfile.ZipFile(zp) as z:
                    z.extractall(zp.parent)
                zp.unlink()
            except Exception:
                pass

        info = find_xview(P["DATA"])
        if not info["valid"]:
            raise RuntimeError("Download failed or dataset structure not recognized")

        DATA_DIR, DS_INFO = P["DATA"], info

IMAGES_DIR = DS_INFO["images_dir"]
GEOJSON_PATH = DS_INFO["geojson"]
LABELS_DIR = DS_INFO["labels_dir"]

# Determine annotation type
if GEOJSON_PATH is not None:
    ANNO_TYPE = "geojson"
elif LABELS_DIR is not None:
    g2 = list(LABELS_DIR.glob("*.geojson"))
    if g2:
        GEOJSON_PATH = g2[0]
        ANNO_TYPE = "geojson"
    else:
        ANNO_TYPE = "txt"
else:
    ANNO_TYPE = "unknown"

print(f"  DATA_DIR  : {DATA_DIR}")
print(f"  Images    : {IMAGES_DIR}")
print(f"  Labels    : {LABELS_DIR}")
print(f"  Anno type : {ANNO_TYPE}")
print(f"  GeoJSON   : {GEOJSON_PATH}")
print("[3.1] Done.")

[3.1] Finding dataset ...
  Scanning /kaggle/input ...
  Top-level folders found:
    - /kaggle/input/datasets
    Checking /kaggle/input/datasets ...
    MATCH FOUND
      Images : /kaggle/input/datasets/hassanmojab/xview-dataset/train_images/train_images
      Labels : /kaggle/input/datasets/hassanmojab/xview-dataset/train_labels
      GeoJSON: /kaggle/input/datasets/hassanmojab/xview-dataset/train_labels/xView_train.geojson
  DATA_DIR  : /kaggle/input/datasets
  Images    : /kaggle/input/datasets/hassanmojab/xview-dataset/train_images/train_images
  Labels    : /kaggle/input/datasets/hassanmojab/xview-dataset/train_labels
  Anno type : geojson
  GeoJSON   : /kaggle/input/datasets/hassanmojab/xview-dataset/train_labels/xView_train.geojson
[3.1] Done.


---
## 4. Annotation Loading & Class Selection


In [32]:
# ══════════════════════════════════════════════════════════════════
#  4.1  Load annotations
# ══════════════════════════════════════════════════════════════════
print("[4.1] Loading annotations ...")
t0 = time.time()

if ANNO_TYPE == "geojson":
    with open(GEOJSON_PATH) as f: raw = json.load(f)
    print(f"  GeoJSON features: {len(raw['features']):,}")
    records = []
    for feat in tqdm(raw["features"], desc="  Parsing"):
        props = feat.get("properties",{})
        coords = props.get("bounds_imcoords","")
        if not coords: continue
        try: x1,y1,x2,y2 = (float(v) for v in coords.split(","))
        except: continue
        w,h = abs(x2-x1), abs(y2-y1)
        records.append({"image_id":props.get("image_id",""),
            "class_id":int(props.get("type_id",-1)),
            "px_x1":min(x1,x2),"px_y1":min(y1,y2),
            "px_x2":max(x1,x2),"px_y2":max(y1,y2),
            "px_w":w,"px_h":h,"px_area":w*h})
    del raw
    bbox_df = pd.DataFrame(records)
else:
    raise NotImplementedError(f"TXT label loading not yet needed — ANNO_TYPE={ANNO_TYPE}")

print(f"  Boxes: {len(bbox_df):,}  Images: {bbox_df['image_id'].nunique()}")
print(f"  Time: {time.time()-t0:.1f}s")
print("[4.1] Done.")


[4.1] Loading annotations ...
  GeoJSON features: 601,937


  Parsing:   0%|          | 0/601937 [00:00<?, ?it/s]

  Boxes: 601,937  Images: 847
  Time: 16.8s
[4.1] Done.


In [33]:
# ══════════════════════════════════════════════════════════════════
#  4.2  Target class selection (Paper Table 3)
# ══════════════════════════════════════════════════════════════════
print("[4.2] Selecting target classes ...")

TARGET_CLS = {18:"Small Car", 17:"Passenger Vehicle",
              20:"Pickup Truck", 21:"Utility Truck"}
CLS_TO_IDX = {18:0, 17:1, 20:2, 21:3}

bbox_df["is_target"] = bbox_df["class_id"].isin(TARGET_CLS)
small_df = bbox_df[
    bbox_df["is_target"]
    & (bbox_df["px_area"] >= MIN_OBJ_PX)
    & (bbox_df["px_area"] <= MAX_OBJ_PX)
].copy()

for cid,cn in TARGET_CLS.items():
    s = small_df[small_df["class_id"]==cid]
    if len(s):
        print(f"  {cn:22s} ID={cid:2d}  {len(s):>8,} objs  "
              f"{s['image_id'].nunique():>4} imgs  "
              f"area {s['px_area'].min():.0f}-{s['px_area'].max():.0f}")
print(f"  Total: {len(small_df):,} objects across {small_df['image_id'].nunique()} images")
print("[4.2] Done.")


[4.2] Selecting target classes ...
  Small Car              ID=18   211,553 objs   692 imgs  area 10-990
  Passenger Vehicle      ID=17     2,950 objs   203 imgs  area 24-819
  Pickup Truck           ID=20     1,102 objs   212 imgs  area 50-735
  Utility Truck          ID=21     3,635 objs   355 imgs  area 32-920
  Total: 219,240 objects across 701 images
[4.2] Done.


---
## 5. Size & Aspect-Ratio Analysis (Paper Figure 3)


In [34]:
# ══════════════════════════════════════════════════════════════════
#  5.1  Figure 3 + Table 3
# ══════════════════════════════════════════════════════════════════
print("[5.1] Plotting distributions ...")
fig, axes = plt.subplots(1,2,figsize=(16,6))
bins_s = np.logspace(0,np.log10(100_000),20)
axes[0].hist(bbox_df["px_area"].clip(upper=100_000),bins=bins_s,alpha=.7,color="steelblue",edgecolor="w",label="All")
axes[0].hist(small_df["px_area"].clip(upper=100_000),bins=bins_s,alpha=.7,color="indianred",edgecolor="w",label="Target")
axes[0].axvline(1000,color="red",ls="--",alpha=.6); axes[0].set_xscale("log")
axes[0].set_xlabel("Area (px)"); axes[0].set_title("(a) Size Distribution"); axes[0].legend()
short=np.minimum(small_df["px_w"],small_df["px_h"])
long_=np.maximum(small_df["px_w"],small_df["px_h"])
ar=(short/long_.replace(0,np.nan)).dropna()
axes[1].hist(ar,bins=np.arange(0,1.05,.1),color="indianred",edgecolor="w")
axes[1].set_xlabel("Aspect Ratio"); axes[1].set_title("(b) Aspect Ratio"); 
plt.tight_layout(); plt.savefig(P["PLOTS"]/"fig3.png",dpi=150,bbox_inches="tight"); plt.show()

# Table 3
rows=[]
for cid,cn in TARGET_CLS.items():
    s=small_df[small_df["class_id"]==cid]
    if len(s): rows.append({"Class":cn,"Images":s["image_id"].nunique(),
        "Objects":len(s),"Min":int(s["px_area"].min()),
        "Median":int(s["px_area"].median()),"Max":int(s["px_area"].max())})
t3=pd.DataFrame(rows); print(t3.to_string(index=False))
t3.to_csv(P["METRICS"]/"table3.csv",index=False)
print("[5.1] Done.")


[5.1] Plotting distributions ...
            Class  Images  Objects  Min  Median  Max
        Small Car     692   211553   10     153  990
Passenger Vehicle     203     2950   24     144  819
     Pickup Truck     212     1102   50     180  735
    Utility Truck     355     3635   32     228  920
[5.1] Done.


---
## 6. Image Tiling (Paper Section 4.1: 512×512)

Tiles all annotated images. Results are cached.


In [35]:
# ══════════════════════════════════════════════════════════════════
#  6.1  Tiling pipeline
# ══════════════════════════════════════════════════════════════════
print("[6.1] Tiling ...")
t0 = time.time()

def tile_pos(h,w,ts=512):
    nr,nc=int(np.ceil(h/ts)),int(np.ceil(w/ts))
    out=[]
    for r in range(nr):
        for c in range(nc):
            y=r*(h-ts)//max(nr-1,1) if nr>1 else 0
            x=c*(w-ts)//max(nc-1,1) if nc>1 else 0
            out.append((min(y,max(0,h-ts)),min(x,max(0,w-ts))))
    return out

def clip_box(bx,ty,tx,ts,mf=.2):
    x1=max(bx["px_x1"]-tx,0); y1=max(bx["px_y1"]-ty,0)
    x2=min(bx["px_x2"]-tx,ts); y2=min(bx["px_y2"]-ty,ts)
    cw,ch=x2-x1,y2-y1
    if cw<=0 or ch<=0: return None
    if bx["px_area"]>0 and cw*ch/bx["px_area"]<mf: return None
    return {"x1":x1,"y1":y1,"x2":x2,"y2":y2,"w":cw,"h":ch,"cid":bx["class_id"]}

img_out=P["TILES"]/"images"; lbl_out=P["TILES"]/"labels"
img_out.mkdir(parents=True,exist_ok=True); lbl_out.mkdir(parents=True,exist_ok=True)
mf_csv=P["TILES"]/"manifest.csv"

if mf_csv.exists() and not FORCE_REDOWNLOAD:
    manifest=pd.read_csv(mf_csv)
    if len(list(img_out.glob("*.png")))>=len(manifest) and len(manifest)>10:
        print(f"  Cached: {len(manifest)} tiles"); manifest_ready=True
    else: manifest_ready=False
else: manifest_ready=False

if not manifest_ready:
    target_imgs=small_df["image_id"].unique()
    if DEBUG_MODE: target_imgs=target_imgs[:30]
    print(f"  Tiling {len(target_imgs)} images ...")
    records=[]
    for img_id in tqdm(target_imgs,desc="  Tiling"):
        ip=IMAGES_DIR/str(img_id)
        if not ip.exists():
            for ext in [".tif",".tiff",".png"]:
                c=IMAGES_DIR/(str(img_id).split(".")[0]+ext)
                if c.exists(): ip=c; break
        if not ip.exists(): continue
        img=cv2.imread(str(ip))
        if img is None: continue
        h,w=img.shape[:2]
        iboxes=small_df[small_df["image_id"]==img_id]
        if len(iboxes)==0: continue
        for idx,(ty,tx) in enumerate(tile_pos(h,w,TILE_SIZE)):
            tile=img[ty:ty+TILE_SIZE,tx:tx+TILE_SIZE]
            if tile.shape[0]<TILE_SIZE or tile.shape[1]<TILE_SIZE:
                pad=np.zeros((TILE_SIZE,TILE_SIZE,3),dtype=np.uint8)
                pad[:tile.shape[0],:tile.shape[1]]=tile; tile=pad
            tboxes=[]
            for _,bx in iboxes.iterrows():
                cb=clip_box(bx,ty,tx,TILE_SIZE)
                if cb: tboxes.append(cb)
            if not tboxes: continue
            tn=f"{Path(img_id).stem}_t{idx:04d}"
            cv2.imwrite(str(img_out/f"{tn}.png"),tile)
            lines=[]
            for b in tboxes:
                cx=(b["x1"]+b["x2"])/2/TILE_SIZE; cy=(b["y1"]+b["y2"])/2/TILE_SIZE
                bw=b["w"]/TILE_SIZE; bh=b["h"]/TILE_SIZE
                lines.append(f"{CLS_TO_IDX.get(b['cid'],0)} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
            (lbl_out/f"{tn}.txt").write_text("\n".join(lines))
            records.append({"tile":tn,"src":img_id,"n_obj":len(tboxes)})
    manifest=pd.DataFrame(records)
    manifest.to_csv(mf_csv,index=False)

print(f"  Tiles: {len(manifest):,}  Objects: {manifest['n_obj'].sum():,}")
print(f"  Time: {time.time()-t0:.1f}s")
print("[6.1] Done.")


[6.1] Tiling ...
  Cached: 668 tiles
  Tiles: 668  Objects: 19,760
  Time: 0.0s
[6.1] Done.


---
## 7. 6-Fold Cross-Validation Split

### Paper Section 4.1
> "We conducted 6-fold cross-validation and report the average performance."

Split is done at the **source-image level** — all tiles from the same raw
image go into the same fold.  This prevents data leakage.

### Why not use val_images?
The xView challenge `val_images/` has **no public annotations**.  The labels
were kept private for the competition.  We can only train/evaluate on the
annotated `train_images/` + GeoJSON data.


In [36]:
# ══════════════════════════════════════════════════════════════════
#  7.1  Generate 6-fold split
# ══════════════════════════════════════════════════════════════════
print("[7.1] Generating 6-fold CV splits ...")

from sklearn.model_selection import KFold

src_images = manifest["src"].unique()
print(f"  Source images: {len(src_images)}")
print(f"  Total tiles : {len(manifest)}")

kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)

fold_splits = {}
for fold_idx, (train_src_idx, val_src_idx) in enumerate(kf.split(src_images)):
    train_srcs = set(src_images[train_src_idx])
    val_srcs   = set(src_images[val_src_idx])

    train_tiles = manifest[manifest["src"].isin(train_srcs)]["tile"].tolist()
    val_tiles   = manifest[manifest["src"].isin(val_srcs)]["tile"].tolist()

    train_objs = manifest[manifest["src"].isin(train_srcs)]["n_obj"].sum()
    val_objs   = manifest[manifest["src"].isin(val_srcs)]["n_obj"].sum()

    fold_splits[fold_idx] = {"train": train_tiles, "val": val_tiles}

    print(f"  Fold {fold_idx}: train={len(train_tiles):>5} tiles ({train_objs:>7,} objs)"
          f"  val={len(val_tiles):>5} tiles ({val_objs:>6,} objs)"
          f"  [src: {len(train_srcs)} / {len(val_srcs)}]")

# Save fold metadata
fold_meta = {f"fold_{i}": {"train":len(s["train"]),"val":len(s["val"])}
             for i,s in fold_splits.items()}
fold_meta["num_folds"] = NUM_FOLDS
fold_meta["seed"] = SEED
with open(P["METRICS"]/"fold_splits.json","w") as f:
    json.dump(fold_meta, f, indent=2)

print(f"\n  Fold metadata saved to {P['METRICS']/'fold_splits.json'}")
print("[7.1] Done.")


[7.1] Generating 6-fold CV splits ...
  Source images: 30
  Total tiles : 668
  Fold 0: train=  605 tiles ( 18,306 objs)  val=   63 tiles ( 1,454 objs)  [src: 25 / 5]
  Fold 1: train=  580 tiles ( 18,533 objs)  val=   88 tiles ( 1,227 objs)  [src: 25 / 5]
  Fold 2: train=  543 tiles ( 15,774 objs)  val=  125 tiles ( 3,986 objs)  [src: 25 / 5]
  Fold 3: train=  542 tiles ( 15,779 objs)  val=  126 tiles ( 3,981 objs)  [src: 25 / 5]
  Fold 4: train=  510 tiles ( 14,334 objs)  val=  158 tiles ( 5,426 objs)  [src: 25 / 5]
  Fold 5: train=  560 tiles ( 16,074 objs)  val=  108 tiles ( 3,686 objs)  [src: 25 / 5]

  Fold metadata saved to /kaggle/working/outputs/metrics/fold_splits.json
[7.1] Done.


In [37]:
# ══════════════════════════════════════════════════════════════════
#  7.2  Compute paper-matched training budget
# ══════════════════════════════════════════════════════════════════
print("[7.2] Computing training budget ...")

# Use fold 0 as reference for size (all folds are similar)
n_train_tiles = len(fold_splits[0]["train"])
iters_per_epoch = int(np.ceil(n_train_tiles / BATCH_SIZE))

if not DEBUG_MODE:
    EPOCHS = max(1, PAPER_TOTAL_ITERS // iters_per_epoch)
    actual_iters = EPOCHS * iters_per_epoch
    print(f"  Paper target    : {PAPER_TOTAL_ITERS:,} iterations")
    print(f"  Train tiles     : {n_train_tiles:,}")
    print(f"  Batch size      : {BATCH_SIZE}")
    print(f"  Iters/epoch     : {iters_per_epoch:,}")
    print(f"  Computed epochs : {EPOCHS}")
    print(f"  Actual iters    : {actual_iters:,}")
    print(f"  Match error     : {abs(actual_iters-PAPER_TOTAL_ITERS)/PAPER_TOTAL_ITERS*100:.1f}%")
else:
    print(f"  DEBUG: {EPOCHS} epoch(s), {iters_per_epoch} iters/epoch")

# Early stopping: DISABLED in paper mode (paper runs full budget)
PATIENCE = 5 if DEBUG_MODE else 999

print(f"  Patience        : {PATIENCE}  ({'debug' if DEBUG_MODE else 'disabled for paper'})")
print("[7.2] Done.")


[7.2] Computing training budget ...
  Paper target    : 94,000 iterations
  Train tiles     : 605
  Batch size      : 8
  Iters/epoch     : 76
  Computed epochs : 1236
  Actual iters    : 93,936
  Match error     : 0.1%
  Patience        : 999  (disabled for paper)
[7.2] Done.


---
## 8. YOLO Dataset Setup & Training

For each fold:
1. Create YOLO directory layout (images/labels train/val)
2. Write `dataset.yaml`
3. Train YOLOv11 with paper hyperparameters
4. Evaluate and save metrics


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  8.1  Setup + Train + Evaluate per fold
# ══════════════════════════════════════════════════════════════════
print("[8.1] Starting fold training loop ...")

def setup_fold_dataset(fold_idx, fold_data, paths):
    """Create YOLO directory layout for one fold."""
    fold_dir = paths["YOLO_DS"] / f"fold_{fold_idx}"
    yaml_p = fold_dir / "dataset.yaml"

    for split in ["train","val"]:
        (fold_dir/"images"/split).mkdir(parents=True,exist_ok=True)
        (fold_dir/"labels"/split).mkdir(parents=True,exist_ok=True)

    # Check cache
    n_existing = sum(1 for _ in (fold_dir/"images"/"train").glob("*.png"))
    if yaml_p.exists() and n_existing > 10:
        print(f"    Fold {fold_idx} dataset cached ({n_existing} train imgs)")
        return yaml_p

    si = paths["TILES"]/"images"
    sl = paths["TILES"]/"labels"

    def lnk(src,dst):
        if dst.exists(): return
        try: os.symlink(src.resolve(),dst)
        except OSError: shutil.copy2(src,dst)

    for split_name in ["train","val"]:
        for tn in fold_data[split_name]:
            ip=si/f"{tn}.png"; lp=sl/f"{tn}.txt"
            if ip.exists(): lnk(ip, fold_dir/"images"/split_name/f"{tn}.png")
            if lp.exists(): lnk(lp, fold_dir/"labels"/split_name/f"{tn}.txt")

    yd = {"path":str(fold_dir.resolve()),"train":"images/train","val":"images/val",
          "nc":4,"names":{0:"Small_Car",1:"Passenger_Vehicle",2:"Pickup_Truck",3:"Utility_Truck"}}
    with open(yaml_p,"w") as f: yaml.dump(yd,f,default_flow_style=False)

    n_tr = sum(1 for _ in (fold_dir/"images"/"train").glob("*.png"))
    n_vl = sum(1 for _ in (fold_dir/"images"/"val").glob("*.png"))
    print(f"    Fold {fold_idx}: {n_tr} train, {n_vl} val images")
    return yaml_p


def train_fold(fold_idx, yaml_path, paths):
    """Train YOLOv11 for one fold."""
    exp_name = f"yolov11_fold{fold_idx}"
    exp_weights = paths["CKPT"] / exp_name / "weights"
    resume_from = None

    # Check for valid checkpoint
    if RESUME_TRAINING and exp_weights.exists():
        last_pt = exp_weights / "last.pt"
        if last_pt.exists():
            try:
                ckpt = torch.load(str(last_pt), map_location="cpu", weights_only=False)
                if ckpt.get("epoch") is not None and ckpt.get("optimizer") is not None:
                    resume_from = str(last_pt)
                    print(f"    Resuming fold {fold_idx} from epoch {ckpt['epoch']}")
                else:
                    print(f"    Invalid checkpoint — deleting")
                    last_pt.unlink()
                    best_pt = exp_weights/"best.pt"
                    if best_pt.exists(): best_pt.unlink()
                del ckpt
            except:
                last_pt.unlink(missing_ok=True)

    args = dict(
        data=str(yaml_path), epochs=EPOCHS, batch=BATCH_SIZE, imgsz=IMG_SIZE,
        lr0=LEARNING_RATE, lrf=0.01, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY,
        flipud=0., fliplr=FLIP_PROB, mosaic=0., mixup=0., scale=0.,
        hsv_h=0., hsv_s=0., hsv_v=0.,
        optimizer="SGD", seed=SEED+fold_idx, deterministic=True,
        workers=2 if DEBUG_MODE else 4, patience=PATIENCE,
        project=str(paths["CKPT"]), name=exp_name, exist_ok=True,
        save=True, save_period=1 if DEBUG_MODE else 10, plots=True, verbose=True,
    )

    # Sanity
    assert "coco" not in args["data"].lower(), f"BUG: data={args['data']}"

    if resume_from:
        model = YOLO(resume_from)
        model.train(resume=True)
    else:
        model = YOLO(YOLO_VARIANT)
        model.train(**args)

    # Verify
    if (paths["CKPT"]/exp_name/"args.yaml").exists():
        with open(paths["CKPT"]/exp_name/"args.yaml") as f:
            used = yaml.safe_load(f)
        if "coco" in str(used.get("data","")).lower():
            print(f"    ERROR: Fold {fold_idx} trained on COCO, not xView!")
            return None

    return model


def evaluate_fold(fold_idx, yaml_path, paths):
    """Evaluate one fold and return metrics dict."""
    best_pt = paths["CKPT"] / f"yolov11_fold{fold_idx}" / "weights" / "best.pt"
    if not best_pt.exists():
        print(f"    No best.pt for fold {fold_idx}")
        return None

    model = YOLO(str(best_pt))
    r = model.val(data=str(yaml_path), imgsz=IMG_SIZE, conf=CONF_THRESHOLD,
                  split="val", plots=True, verbose=False)

    met = {
        "fold": fold_idx,
        "AP50": round(float(r.box.map50)*100, 2),
        "AP50_95": round(float(r.box.map)*100, 2),
        "Precision": round(float(r.box.mp)*100, 2),
        "Recall": round(float(r.box.mr)*100, 2),
    }
    p,rc = met["Precision"]/100, met["Recall"]/100
    met["F1"] = round(2*p*rc/(p+rc)*100, 2) if (p+rc)>0 else 0

    print(f"    Fold {fold_idx}: AP50={met['AP50']:.2f}  AP50:95={met['AP50_95']:.2f}  "
          f"F1={met['F1']:.2f}")
    return met


# ══════════════════════════════════════════════════════════════════
#  Determine which folds to run
# ══════════════════════════════════════════════════════════════════
if isinstance(CURRENT_FOLD, int):
    folds_to_run = [CURRENT_FOLD]
elif CURRENT_FOLD == "all":
    folds_to_run = list(range(NUM_FOLDS))
else:
    folds_to_run = [0]

print(f"  Folds to run: {folds_to_run}")
print(f"  Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, LR: {LEARNING_RATE}")
print(f"  Model: {YOLO_VARIANT}")
print(f"  Patience: {PATIENCE}")

all_metrics = []

for fi in folds_to_run:
    print(f"\n{'='*60}")
    print(f"  FOLD {fi}/{NUM_FOLDS-1}")
    print(f"{'='*60}")

    # Setup
    yaml_p = setup_fold_dataset(fi, fold_splits[fi], P)

    # Train
    if not SKIP_TRAINING:
        t0 = time.time()
        train_fold(fi, yaml_p, P)
        print(f"    Training time: {(time.time()-t0)/60:.1f} min")

    # Evaluate
    met = evaluate_fold(fi, yaml_p, P)
    if met:
        all_metrics.append(met)
        # Save per-fold result
        pd.DataFrame([met]).to_csv(
            P["METRICS"]/f"fold{fi}_results.csv", index=False)

print(f"\n  Completed {len(all_metrics)} fold(s)")
print("[8.1] Done.")


[8.1] Starting fold training loop ...
  Folds to run: [0]
  Epochs: 1236, Batch: 8, LR: 0.000101
  Model: yolo11m.pt
  Patience: 999

  FOLD 0/5
    Fold 0 dataset cached (605 train imgs)
    Invalid checkpoint — deleting
Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/outputs/yolo_dataset/fold_0/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1236, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width

---
## 9. Results Aggregation (mean ± std)

The paper reports mean and standard deviation across 6 folds.


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  9.1  Aggregate fold results
# ══════════════════════════════════════════════════════════════════
print("[9.1] Aggregating results ...")

# Load any previously saved fold results
saved_results = []
for fi in range(NUM_FOLDS):
    fp = P["METRICS"]/f"fold{fi}_results.csv"
    if fp.exists():
        saved_results.append(pd.read_csv(fp).iloc[0].to_dict())

if saved_results:
    results_df = pd.DataFrame(saved_results)
    n_folds_done = len(results_df)

    print(f"  Results from {n_folds_done}/{NUM_FOLDS} folds:\n")

    # Per-fold table
    print(results_df.to_string(index=False))

    # Mean ± std
    metrics = ["AP50","AP50_95","Precision","Recall","F1"]
    print(f"\n  {'Metric':12s}  {'Mean':>8s}  {'Std':>7s}  Paper (mean±std)")
    print("  " + "-"*55)
    paper = {"AP50":(69.47,3.07),"AP50_95":(29.58,1.40),"F1":(73.97,1.81)}
    for m in metrics:
        if m in results_df.columns:
            mean_v = results_df[m].mean()
            std_v  = results_df[m].std() if n_folds_done > 1 else 0
            p_str = ""
            if m in paper:
                pm,ps = paper[m]
                p_str = f"{pm:.2f} ± {ps:.2f}"
            print(f"  {m:12s}  {mean_v:8.2f}  {std_v:7.2f}  {p_str}")

    # Save aggregated
    agg = {m: {"mean": results_df[m].mean(), "std": results_df[m].std()}
           for m in metrics if m in results_df.columns}
    agg["n_folds"] = n_folds_done
    with open(P["METRICS"]/"aggregated_results.json","w") as f:
        json.dump(agg, f, indent=2)
    results_df.to_csv(P["METRICS"]/"all_folds_results.csv", index=False)
    print(f"\n  Saved: aggregated_results.json, all_folds_results.csv")

    if n_folds_done < NUM_FOLDS:
        print(f"\n  NOTE: Only {n_folds_done}/{NUM_FOLDS} folds completed.")
        print(f"  Run remaining folds by changing CURRENT_FOLD = {n_folds_done}")
        print(f"  and re-running the notebook.")

    if DEBUG_MODE:
        print("\n  DEBUG results — not paper-comparable.")
        print("  Set DEBUG_MODE=False and run all 6 folds.")
else:
    print("  No fold results found. Train at least one fold first.")

print("[9.1] Done.")


---
## 10. Visualization


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  10.1  Prediction overlays (from latest fold)
# ══════════════════════════════════════════════════════════════════
print("[10.1] Prediction overlays ...")

last_fold = folds_to_run[-1] if folds_to_run else 0
best_pt = P["CKPT"]/f"yolov11_fold{last_fold}"/"weights"/"best.pt"
fold_ds = P["YOLO_DS"]/f"fold_{last_fold}"

if not best_pt.exists():
    print("  No model — skipping.")
else:
    vm = YOLO(str(best_pt))
    val_imgs = list((fold_ds/"images"/"val").glob("*.png"))
    if not val_imgs:
        print("  No val images.")
    else:
        sel = random.sample(val_imgs, min(6,len(val_imgs)))
        nr = (len(sel)+2)//3
        fig,axes = plt.subplots(nr,3,figsize=(20,7*nr))
        axes = np.array(axes).flatten()
        for i,ip in enumerate(sel):
            ax=axes[i]
            img=cv2.cvtColor(cv2.imread(str(ip)),cv2.COLOR_BGR2RGB)
            lp=fold_ds/"labels"/"val"/(ip.stem+".txt")
            ng=0
            if lp.exists():
                for ln in lp.read_text().strip().split("\n"):
                    p=ln.split()
                    if len(p)==5:
                        _,cx,cy,w,h=map(float,p)
                        cv2.rectangle(img,(int((cx-w/2)*512),int((cy-h/2)*512)),
                                      (int((cx+w/2)*512),int((cy+h/2)*512)),(0,255,0),2)
                        ng+=1
            r=vm.predict(str(ip),conf=CONF_THRESHOLD,verbose=False)
            np_=0
            if r and len(r[0].boxes):
                for b in r[0].boxes:
                    x1,y1,x2,y2=b.xyxy[0].cpu().numpy().astype(int)
                    cv2.rectangle(img,(x1,y1),(x2,y2),(255,0,0),2); np_+=1
            ax.imshow(img); ax.set_title(f"GT:{ng} Pred:{np_}"); ax.axis("off")
        for j in range(i+1,len(axes)): axes[j].axis("off")
        plt.suptitle(f"Fold {last_fold}: Green=GT, Red=Pred",fontsize=14)
        plt.tight_layout()
        plt.savefig(P["PLOTS"]/"predictions.png",dpi=150,bbox_inches="tight")
        plt.show()
print("[10.1] Done.")


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  10.2  Training curves
# ══════════════════════════════════════════════════════════════════
print("[10.2] Training curves ...")
csv_p = P["CKPT"]/f"yolov11_fold{last_fold}"/"results.csv"
if not csv_p.exists():
    print("  No results.csv")
else:
    df=pd.read_csv(csv_p); df.columns=[c.strip() for c in df.columns]
    fig,axes=plt.subplots(2,3,figsize=(18,10))
    for i,(t,v,ttl) in enumerate([("train/box_loss","val/box_loss","Box"),
        ("train/cls_loss","val/cls_loss","Cls"),("train/dfl_loss","val/dfl_loss","DFL")]):
        ax=axes[0,i]
        if t in df: ax.plot(df["epoch"],df[t],label="Train",lw=2)
        if v in df: ax.plot(df["epoch"],df[v],label="Val",lw=2)
        ax.set_title(ttl); ax.legend(); ax.grid(alpha=.3)
    ax=axes[1,0]
    for c in [x for x in df.columns if "map" in x.lower()]:
        ax.plot(df["epoch"],df[c],label=c,lw=2)
    ax.set_title("mAP"); ax.legend(fontsize=8)
    ax=axes[1,1]
    if "metrics/precision(B)" in df:
        ax.plot(df["epoch"],df["metrics/precision(B)"],label="P")
        ax.plot(df["epoch"],df["metrics/recall(B)"],label="R")
    ax.set_title("P & R"); ax.legend()
    ax=axes[1,2]
    for c in [x for x in df.columns if "lr" in x.lower()]:
        ax.plot(df["epoch"],df[c],label=c)
    ax.set_title("LR"); ax.legend(fontsize=8)
    for a in axes.flatten(): a.set_xlabel("Epoch"); a.grid(alpha=.3)
    plt.suptitle(f"Fold {last_fold} Training Curves",fontsize=14)
    plt.tight_layout()
    plt.savefig(P["PLOTS"]/"curves.png",dpi=150,bbox_inches="tight"); plt.show()
print("[10.2] Done.")


---
## 11. Error Analysis (size & density breakdown)


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  11.1  Error analysis
# ══════════════════════════════════════════════════════════════════
print("[11.1] Error analysis ...")
if not best_pt.exists():
    print("  No model."); 
else:
    em=YOLO(str(best_pt))
    vi=list((fold_ds/"images"/"val").glob("*.png"))
    vl=fold_ds/"labels"/"val"
    ns=min(50 if DEBUG_MODE else 200,len(vi))
    samp=random.sample(vi,ns)
    S={"tp":0,"fp":0,"fn":0,"gt":0,
       "sz":defaultdict(lambda:{"gt":0,"tp":0}),
       "dn":defaultdict(lambda:{"gt":0,"tp":0})}
    for ip in tqdm(samp,desc="  Analysis"):
        lp=vl/(ip.stem+".txt"); gt=[]
        if lp.exists():
            for ln in lp.read_text().strip().split("\n"):
                p=ln.split()
                if len(p)==5:
                    _,cx,cy,w,h=map(float,p)
                    gt.append({"cx":cx,"cy":cy,"w":w,"h":h,"a":w*h*512*512})
        r=em.predict(str(ip),conf=CONF_THRESHOLD,verbose=False); pds=[]
        if r and len(r[0].boxes):
            for b in r[0].boxes:
                x1,y1,x2,y2=b.xyxy[0].cpu().numpy()/512
                pds.append({"x1":x1,"y1":y1,"x2":x2,"y2":y2})
        S["gt"]+=len(gt)
        mg,mp=set(),set()
        for pi,p in enumerate(pds):
            bi,bv=-1,0
            for gi,g in enumerate(gt):
                if gi in mg: continue
                gx1,gy1=g["cx"]-g["w"]/2,g["cy"]-g["h"]/2
                gx2,gy2=g["cx"]+g["w"]/2,g["cy"]+g["h"]/2
                ix=max(0,min(p["x2"],gx2)-max(p["x1"],gx1))
                iy=max(0,min(p["y2"],gy2)-max(p["y1"],gy1))
                inter=ix*iy; un=(p["x2"]-p["x1"])*(p["y2"]-p["y1"])+g["w"]*g["h"]-inter
                iou=inter/max(un,1e-9)
                if iou>bv: bv=iou;bi=gi
            if bv>=.5: mg.add(bi);mp.add(pi);S["tp"]+=1
        S["fp"]+=len(pds)-len(mp); S["fn"]+=len(gt)-len(mg)
        for gi,g in enumerate(gt):
            a=g["a"]
            bn="<100" if a<100 else "100-300" if a<300 else "300-600" if a<600 else "600+"
            S["sz"][bn]["gt"]+=1
            if gi in mg: S["sz"][bn]["tp"]+=1
        d="sparse" if len(gt)<=5 else "moderate" if len(gt)<=20 else "dense"
        S["dn"][d]["gt"]+=len(gt); S["dn"][d]["tp"]+=len(mg)
    pr=S["tp"]/max(S["tp"]+S["fp"],1); rc=S["tp"]/max(S["tp"]+S["fn"],1)
    f1=2*pr*rc/max(pr+rc,1e-9)
    print(f"  Precision={pr:.3f}  Recall={rc:.3f}  F1={f1:.3f}")
    print(f"  By size:")
    for b in ["<100","100-300","300-600","600+"]:
        s=S["sz"][b]
        if s["gt"]: print(f"    {b:10s} {s['gt']:5d} GT  recall={s['tp']/s['gt']:.3f}")
    print(f"  By density:")
    for d in ["sparse","moderate","dense"]:
        s=S["dn"][d]
        if s["gt"]: print(f"    {d:10s} {s['gt']:5d} GT  recall={s['tp']/s['gt']:.3f}")
    with open(P["METRICS"]/"error_analysis.json","w") as f:
        json.dump({"precision":pr,"recall":rc,"f1":f1},f,indent=2)
print("[11.1] Done.")


---
## 12. Summary & Limitations

### What this notebook implements
| Paper aspect | Status | Notes |
|---|---|---|
| 6-fold CV | Done | Source-image-level splits |
| 512×512 tiling | Done | With overlap, box clipping |
| YOLOv11 ~25M params | Done | `yolo11m.pt` = 20M (closest available) |
| ~94K iterations | Done | Computed from dataset size |
| SGD, LR=1.01e-4 | Done | Paper Table 5 |
| Flip-only augmentation | Done | mosaic/mixup/hsv disabled |
| No early stopping | Done | patience=999 in paper mode |
| Mean ± std reporting | Done | Across completed folds |

### Known limitations
| Aspect | Detail |
|---|---|
| 8-channel imagery | xView is 8-band WorldView-3, but Kaggle serves RGB-only TIF. YOLO requires 3ch anyway. |
| Model size | `yolo11m.pt` = 20M params vs paper's 25.37M. No public `yolo11` with exact 25M. |
| SkySat/DOTA | Transfer learning experiments not included. |
| Ablation studies | Anchor sensitivity, attention analysis not implemented. |
| Kaggle time limit | 6 folds × ~3h = ~18h. Run one fold per session. |


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  12.1  Save summary
# ══════════════════════════════════════════════════════════════════
print("[12.1] Saving summary ...")
summary = {
    "paper": "Yuan et al. 2026, ESWA 307",
    "model": YOLO_VARIANT,
    "debug": DEBUG_MODE,
    "env": RUN_ENV,
    "folds": NUM_FOLDS,
    "folds_completed": len(all_metrics) if 'all_metrics' in dir() else 0,
    "epochs": EPOCHS,
    "batch": BATCH_SIZE,
    "lr": LEARNING_RATE,
    "momentum": MOMENTUM,
    "optimizer": "SGD",
    "patience": PATIENCE,
    "tile_size": TILE_SIZE,
    "gpu": HW["name"],
    "timestamp": datetime.now().isoformat(),
}
with open(P["METRICS"]/"summary.json","w") as f:
    json.dump(summary,f,indent=2)

print(f"  Saved: {P['METRICS']/'summary.json'}")
print(f"\n  {'='*50}")
print(f"  COMPLETE")
print(f"  {'='*50}")
print(f"  Mode  : {'DEBUG' if DEBUG_MODE else 'PAPER'}")
print(f"  Model : {YOLO_VARIANT}")
print(f"  Folds : {len(all_metrics) if 'all_metrics' in dir() else '?'}/{NUM_FOLDS}")

if DEBUG_MODE:
    print(f"\n  Set DEBUG_MODE=False for paper results.")
    print(f"  Set CURRENT_FOLD='all' to train all 6 folds.")
    print(f"  Or run folds one at a time: CURRENT_FOLD=0, then 1, etc.")

print("[12.1] Done.")
